# Atividade 03 — Agrupamento (K-Means, AGNES e DBSCAN)

**Tema:** segmentar assinantes de um **serviço de streaming de vídeo**.

Comparamos três algoritmos de clustering e interpretamos os perfis encontrados.

> Rode no Google Colab — dependências instaladas na primeira célula.

In [ ]:
!pip install -q scikit-learn pandas matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
dados = pd.DataFrame({
    "Assinante": [f"U{i:02d}" for i in range(1, 21)],
    "HorasMes": [120, 110, 100, 90, 85, 70, 65, 60, 55, 50,
                 30, 28, 25, 22, 20, 15, 12, 10, 8, 5],
    "Dispositivos": [4, 4, 3, 3, 3, 2, 2, 2, 2, 2,
                     1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    "PerfisFamilia": [4, 3, 3, 2, 2, 2, 2, 1, 1, 1,
                        1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
    "Downloads": [40, 35, 30, 25, 20, 15, 12, 10, 8, 5,
                    2, 2, 1, 1, 0, 0, 0, 0, 0, 0],
    "Cancelamentos": [0, 0, 0, 1, 0, 1, 0, 1, 1, 2,
                      2, 3, 2, 3, 4, 3, 4, 5, 4, 6],
    "PlanoPremium": [1, 1, 1, 1, 0, 1, 0, 0, 0, 0,
                       0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
})
dados

In [ ]:
atributos = ["HorasMes", "Dispositivos", "PerfisFamilia", "Downloads", "Cancelamentos", "PlanoPremium"]
X = dados[atributos]
X.head()

In [ ]:
X_norm = StandardScaler().fit_transform(X)
X_norm[:3]

In [ ]:
dados["Cluster_KMeans"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_norm)
dados[["Assinante", "Cluster_KMeans"]]

In [ ]:
dados["Cluster_AGNES"] = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(X_norm)
dados[["Assinante", "Cluster_AGNES"]]

In [ ]:
dados["Cluster_DBSCAN"] = DBSCAN(eps=1.5, min_samples=2).fit_predict(X_norm)
dados[["Assinante", "Cluster_DBSCAN"]]

In [ ]:
dados[["Assinante"] + atributos + ["Cluster_KMeans", "Cluster_AGNES", "Cluster_DBSCAN"]]

In [ ]:
def calcular_silhouette(labels):
    grupos = set(labels)
    reais = [g for g in grupos if g != -1]
    if len(reais) < 2:
        return None
    return silhouette_score(X_norm, labels)

pd.DataFrame({
    "Algoritmo": ["K-Means", "AGNES", "DBSCAN"],
    "Silhouette": [
        calcular_silhouette(dados["Cluster_KMeans"]),
        calcular_silhouette(dados["Cluster_AGNES"]),
        calcular_silhouette(dados["Cluster_DBSCAN"]),
    ]
})

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_norm)
dados["PCA1"], dados["PCA2"] = coords[:, 0], coords[:, 1]
dados.head()

In [ ]:
def plotar_clusters(coluna, titulo):
    plt.figure(figsize=(7, 5))
    plt.scatter(dados["PCA1"], dados["PCA2"], c=dados[coluna], s=80, edgecolors="k")
    for i, nome in enumerate(dados["Assinante"]):
        plt.text(dados["PCA1"][i], dados["PCA2"][i], nome, fontsize=8)
    plt.title(titulo)
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.show()

plotar_clusters("Cluster_KMeans", "K-Means — assinantes de streaming")
plotar_clusters("Cluster_AGNES", "AGNES — assinantes de streaming")
plotar_clusters("Cluster_DBSCAN", "DBSCAN — assinantes de streaming")

In [ ]:
perfil_kmeans = dados.groupby("Cluster_KMeans")[atributos].mean()
perfil_kmeans

In [ ]:
for cluster, linha in perfil_kmeans.iterrows():
    print(f"\nCluster {cluster} — K-Means")
    print(f"  Horas/mês: {linha['HorasMes']:.1f}")
    print(f"  Dispositivos: {linha['Dispositivos']:.1f}")
    print(f"  Perfis família: {linha['PerfisFamilia']:.1f}")
    print(f"  Downloads: {linha['Downloads']:.1f}")
    print(f"  Cancelamentos: {linha['Cancelamentos']:.1f}")
    print(f"  Plano premium: {linha['PlanoPremium']:.1%}")

In [ ]:
def interpretar_clusters(coluna):
    perfis = dados.groupby(coluna)[atributos].mean()
    for cluster, linha in perfis.iterrows():
        print("=" * 55)
        nome = "Ruído (DBSCAN)" if cluster == -1 else f"Grupo {cluster}"
        tags = []
        if linha["HorasMes"] >= 80: tags.append("heavy users")
        elif linha["HorasMes"] <= 20: tags.append("uso esporádico")
        if linha["PlanoPremium"] >= 0.5: tags.append("maioria no plano premium")
        if linha["Cancelamentos"] >= 3: tags.append("alto risco de churn")
        elif linha["Cancelamentos"] <= 1: tags.append("baixo risco de churn")
        if linha["PerfisFamilia"] >= 2: tags.append("famílias grandes")
        print(f"{nome}: {', '.join(tags) if tags else 'perfil intermediário'}")

print("K-Means")
interpretar_clusters("Cluster_KMeans")
print("\nAGNES")
interpretar_clusters("Cluster_AGNES")
print("\nDBSCAN")
interpretar_clusters("Cluster_DBSCAN")

## Conclusão

Os três algoritmos revelam perfis distintos de assinantes: **heavy users** com plano premium, usuários intermediários e grupos com **alto risco de cancelamento**. O DBSCAN ainda pode marcar casos atípicos como ruído (-1).